# FightFlow — Colab Trainer

Run this notebook top-to-bottom at the start of every Colab session.
Training outputs are saved to Google Drive and persist after the session ends.

> **First time?** Follow the setup guide in the README before running this.

## 1. Check GPU
Make sure Colab has assigned you a GPU. If `CUDA available: False`, go to
**Runtime → Change runtime type → T4 GPU** and re-run.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

## 2. Mount Google Drive
Your videos, clip data, and training outputs all live on Drive so they persist between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone / update repo
Pulls the latest code from GitHub. Already cloned? It just runs `git pull`.

In [ ]:
import os

REPO_URL = "https://github.com/krishmula/fightflow.git"
REPO_DIR = "/content/fightflow"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print(f"Repo ready at {REPO_DIR}")

## 4. Symlink data and runs from Drive
Instead of changing any config files, we symlink `data/` and `runs/` from Drive
into the repo root. All relative paths in `hparams.yaml` and `data_config.yaml` work unchanged.

In [ ]:
DRIVE_ROOT  = "/content/drive/MyDrive/fightflow"
DRIVE_DATA  = f"{DRIVE_ROOT}/data"
DRIVE_RUNS  = f"{DRIVE_ROOT}/runs"

# Create runs folder on Drive if it doesn't exist yet
os.makedirs(DRIVE_RUNS, exist_ok=True)

# Remove any stale symlinks or empty dirs Colab may have created
for subdir in ["data", "runs"]:
    target = f"{REPO_DIR}/{subdir}"
    if os.path.islink(target):
        os.unlink(target)
    elif os.path.isdir(target) and not os.listdir(target):
        os.rmdir(target)

!ln -s {DRIVE_DATA} {REPO_DIR}/data
!ln -s {DRIVE_RUNS} {REPO_DIR}/runs

# Verify
print("data/ →", os.readlink(f"{REPO_DIR}/data"))
print("runs/ →", os.readlink(f"{REPO_DIR}/runs"))
print("\nFiles visible in data/:")
!ls {REPO_DIR}/data

## 5. Install dependencies
Takes ~1 minute. Cached by Colab for the session.

In [ ]:
%pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

## 6. Prepare clip data
Extracts frame clips from your videos and writes them to `data/processed/clips/` on Drive.

**Only run this once** (or after you add new annotations). Skip if clips already exist on Drive.

In [ ]:
import os

clips_dir   = f"{DRIVE_DATA}/processed/clips"
clips_exist = os.path.isdir(clips_dir) and len(os.listdir(clips_dir)) > 0

if clips_exist:
    n = len(os.listdir(clips_dir))
    print(f"Clips already exist on Drive ({n} clips). Skipping prepare_data.")
    print("Delete data/processed/clips/ from Drive if you want to regenerate.")
else:
    print("No clips found — running prepare_data. This may take a few minutes...")
    %cd {REPO_DIR}
    !python main.py --model cnn_lstm --task prepare_data
    print("Done. Clips saved to Drive.")

## 7. Train
Runs the full training loop. Checkpoints and metrics are saved to
`runs/cnn_lstm/<run_name>/` on Drive in real time.

Tweak any hyperparameters below by passing CLI flags — they override `hparams.yaml`.

In [ ]:
%cd {REPO_DIR}

# Add / remove flags to override hparams.yaml defaults.
# Examples:
#   --epochs 30
#   --lr 0.0005
#   --freeze-backbone false
#   --cnn-backbone vgg_16

!python main.py --model cnn_lstm --task train

## 8. (Optional) Evaluate a checkpoint
Run test or validation against a saved checkpoint.

In [ ]:
# List available runs
!ls {DRIVE_RUNS}/cnn_lstm/

In [ ]:
%cd {REPO_DIR}

RUN_NAME = "REPLACE_WITH_RUN_FOLDER_NAME"  # e.g. cnn_lstm_20240508_143012

!python main.py --model cnn_lstm --task test \
    --checkpoint runs/cnn_lstm/{RUN_NAME}/checkpoints/best.pt \
    --report-dir runs/cnn_lstm/{RUN_NAME}/reports